# Temperature interpolation and gap filling

This notebook demonstrates two gap-filling layers: simple one-dimensional interpolation
and the hourly temperature gap workflow that uses daily temperature-curve structure.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from chillPy import interpolate_gaps, interpolate_gaps_hourly, stack_hourly_temps

DATA = Path("examples/data")
daily = pd.read_csv(DATA / "synthetic_daily_weather.csv").query("Year == 2020 and Month == 1 and Day <= 12")
hourly = stack_hourly_temps(daily, latitude=42.0)["hourtemps"]
hourly.head()

In [ ]:
series = pd.Series([2.0, np.nan, np.nan, 8.0, np.nan, 10.0], name="example")
filled = interpolate_gaps(series)
pd.DataFrame({"original": series, "filled": filled["interp"], "was_missing": filled["missing"]})

Create realistic hourly gaps, including a multi-hour gap that crosses midnight.

In [ ]:
hourly_with_gaps = hourly.copy()
gap_mask = (
    ((hourly_with_gaps["Day"] == 4) & (hourly_with_gaps["Hour"].isin([10, 11, 12, 13])))
    | ((hourly_with_gaps["Day"] == 6) & (hourly_with_gaps["Hour"].isin([22, 23])))
    | ((hourly_with_gaps["Day"] == 7) & (hourly_with_gaps["Hour"].isin([0, 1, 2])))
)
hourly_with_gaps.loc[gap_mask, "Temp"] = np.nan

filled_hourly = interpolate_gaps_hourly(
    hourly_with_gaps,
    latitude=42.0,
    minimum_values_for_solving=12,
    runn_mean_test_diff=999,
)["weather"]

filled_hourly.loc[gap_mask.to_numpy(), ["Year", "Month", "Day", "Hour", "Temp_measured", "Temp"]].head(12)

In [ ]:
comparison = filled_hourly.copy()
comparison["timestamp"] = pd.to_datetime(
    comparison[["Year", "Month", "Day"]].astype(int)
) + pd.to_timedelta(comparison["Hour"].astype(int), unit="h")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(comparison["timestamp"], comparison["Temp"], label="filled", color="black")
ax.scatter(
    comparison["timestamp"],
    comparison["Temp_measured"],
    label="measured",
    color="tab:blue",
    s=12,
)
ax.set_ylabel("Temperature (deg C)")
ax.set_title("Hourly gap filling")
ax.legend(frameon=False)
fig.autofmt_xdate()
fig